# 06 · Custom OCR on AWS: SageMaker + S3

**Notebook 6 of 6:** the "on AWS" half of the brief: Task 1.2 (SageMaker notebook instance with S3 access), training on SageMaker (Task 3.1), and section 6 (automation and deployment).

Everything goes through two scripts, so the same steps work from this notebook, a terminal, or CI:

| Script | Does |
|---|---|
| `aws/s3_sync.py` | mirrors `datasets/`, `model/`, `results/` to S3 (S3 plays Google Drive's role) |
| `aws/sm_jobs.py` | ECR image, notebook instance, training job, serverless endpoint, batch job, cleanup |

**One Docker image does everything.** `docker run <img> serve` is the web app *and* a SageMaker endpoint (it implements `/ping` + `/invocations`), `train` is a SageMaker training job, and `ocr` is the batch CLI.

```
 laptop / Colab ──push──► S3 bucket ◄──────────── SageMaker notebook (Task 1.2)
                           │   ▲                         │
               datasets/yolo   │ model.tar.gz            │ starts
                           ▼   │                         ▼
                 Training job (YOLOv3, spot GPU) ──► Serverless endpoint ──► CSV per photo
                 Processing job: s3://…/inbox/*.jpg ──► s3://…/results/batch/*.csv + .xlsx
```

`DRY_RUN = True` (default) prints every AWS request without sending it, so this notebook runs without an AWS account. Set it to `False`, fill in `BUCKET` and `ROLE`, and it runs for real.

In [1]:
# --- 0. Settings -------------------------------------------------------------------------------
import sys, os, subprocess
from pathlib import Path
here = Path.cwd().resolve()
REPO = here.parent if here.name == "notebooks" else here
sys.path[:0] = [str(REPO), str(REPO / "aws"), str(REPO / "tools")]

DRY_RUN = True                                                     # False = really call AWS
BUCKET  = "my-custom-ocr-bucket"                                   # globally unique S3 bucket name
ROLE    = "arn:aws:iam::123456789012:role/SageMakerExecutionRole"  # IAM role SageMaker assumes
REGION  = "us-east-1"
os.environ.setdefault("AWS_DEFAULT_REGION", REGION)

def run(script, *args):
    """Run one of the aws/ scripts and show its output."""
    cmd = [sys.executable, str(REPO / "aws" / script), *(["--dry-run"] if DRY_RUN else []), *args]
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
    print((r.stdout + r.stderr).strip()[:4000])
print("repo:", REPO, "| dry run:", DRY_RUN)

repo: C:\Users\ARKO\Downloads\custom-ocr-lab-reports-main\custom-ocr-lab-reports-main | dry run: True


## Step 1 · S3: the project's storage
The bucket is created **private and encrypted**: the raw photos show a patient's details, so all public access is blocked. Uploads are incremental (unchanged files are skipped by size + MD5).

In [2]:
# --- 1.1 Create the bucket and push data + weights --------------------------------------------
if DRY_RUN:
    run("s3_sync.py", "--bucket", BUCKET, "--region", REGION, "push", "datasets", "model")
else:
    run("s3_sync.py", "--bucket", BUCKET, "--region", REGION, "create")
    run("s3_sync.py", "--bucket", BUCKET, "--region", REGION, "push", "datasets", "model")
    run("s3_sync.py", "--bucket", BUCKET, "--region", REGION, "ls")

(dry run: cannot list s3://my-custom-ocr-bucket/custom-ocr/datasets/ [NoCredentialsError], treating it as empty)
datasets  uploaded 305, unchanged 0 (dry run)
(dry run: cannot list s3://my-custom-ocr-bucket/custom-ocr/model/ [NoCredentialsError], treating it as empty)
model     uploaded 3, unchanged 0 (dry run)


*(Without AWS credentials the dry run cannot list the bucket, so it plans the upload as if the bucket were empty. Every later step prints its full API request offline.)*

## Step 2 · The Docker image in Amazon ECR
`--gpu` builds with CUDA torch for training; the default CPU image (~2 GB) is used for the endpoint and batch jobs.

In [3]:
run("sm_jobs.py", "--region", REGION, "image", "--gpu")
run("sm_jobs.py", "--region", REGION, "image")

[dry-run] docker build --build-arg TORCH_INDEX=https://download.pytorch.org/whl/cu128 -t 123456789012.dkr.ecr.us-east-1.amazonaws.com/custom-ocr:gpu C:\Users\ARKO\Downloads\custom-ocr-lab-reports-main\custom-ocr-lab-reports-main 
[dry-run] docker push 123456789012.dkr.ecr.us-east-1.amazonaws.com/custom-ocr:gpu


[dry-run] docker build -t 123456789012.dkr.ecr.us-east-1.amazonaws.com/custom-ocr:cpu C:\Users\ARKO\Downloads\custom-ocr-lab-reports-main\custom-ocr-lab-reports-main 
[dry-run] docker push 123456789012.dkr.ecr.us-east-1.amazonaws.com/custom-ocr:cpu


## Step 3 · Task 1.2: SageMaker notebook instance
`aws/notebook_lifecycle_on_start.sh` runs at every start: it installs Tesseract, creates a `Python (custom-ocr)` kernel with Ultralytics/OpenCV/pytesseract/boto3, and **stops the instance after 60 idle minutes** so a forgotten notebook does not keep billing. The instance's role gives it access to the bucket; inside it, `python aws/s3_sync.py --bucket <B> pull` fetches the data.

In [4]:
run("sm_jobs.py", "--region", REGION, "notebook", "--role", ROLE, "--bucket", BUCKET)

[dry-run] sagemaker.create_notebook_instance_lifecycle_config(
{
  "NotebookInstanceLifecycleConfigName": "custom-ocr-on-start",
  "OnStart": [
    {
      "Content": "IyEvYmluL2Jhc2gKIyBTYWdlTWFrZXIgbm90ZWJvb2staW5zdGFuY2UgbGlmZWN5Y2xlIGNvbmZpZ3VyYXRpb24gKCJTdGFydCBub3RlYm9vayIgc2NyaXB0KSAtIGJyaWVmIFRhc2sgMS4yLgojIFJ1bnMgYXMgcm9vdCBldmVyeSB0aW1lIHRoZSBpbnN0YW5jZSBzdGFydHMsIHNvIHRoZSBlbnZpcm9ubWVudCBzdXJ2aXZlcyBzdG9wL3N0YXJ0OgojICAgKiBpbnN0YWxscyB0aGUgVGVzc2VyYWN0IE9DUiBlbmdpbmUKIyAgICogY3JlYXRlcyBhICJjdXN0b20tb2NyIiBKdXB5dGVyIGtlcm5lbCB3aXRoIFB5VG9yY2gsIFVsdHJhbHl0aWNzLCBPcGVuQ1YsIHB5dGVzc2VyYWN0LCBib3RvMwojICAgKiBhdXRvLXN0b3BzIHRoZSBpbnN0YW5jZSBhZnRlciA2MCBpZGxlIG1pbnV0ZXMgKGNvc3QgY29udHJvbCkKIyBQYXN0ZSBpdCBpbjogU2FnZU1ha2VyIGNvbnNvbGUgLT4gTm90ZWJvb2sgLT4gTGlmZWN5Y2xlIGNvbmZpZ3VyYXRpb25zIC0+IENyZWF0ZSAtPiBTdGFydCBub3RlYm9vay4KIyAoQ3JlYXRlIGl0IHdpdGg6IGF3cyBzYWdlbWFrZXIgY3JlYXRlLW5vdGVib29rLWluc3RhbmNlLWxpZmVjeWNsZS1jb25maWcgLi4uIHZpYSBhd3Mvc21fam9icy5weSBub3RlYm9vaykKc2V0IC1lCgojIC0t

## Step 4 · Task 3.1 on SageMaker: train YOLOv3
A managed **spot** training job on one `ml.g4dn.xlarge` (T4 GPU). The container reads the `training` channel (`s3://…/datasets/yolo/`), trains with exactly the same augmentation as `tools/train.py`, and writes `best.pt` to `/opt/ml/model`, which SageMaker uploads as `model.tar.gz`. `fetch` waits for it and saves `model/sagemaker_best.pt`.

In [5]:
run("sm_jobs.py", "--region", REGION, "train", "--role", ROLE, "--bucket", BUCKET, "--weights", "yolov3u.pt", "--epochs", "100")
# when it finishes:  run("sm_jobs.py", "fetch", "--job", "<job name printed above>")

[dry-run] sagemaker.create_training_job(
{
  "TrainingJobName": "custom-ocr-yolov3u-20261025-164534",
  "AlgorithmSpecification": {
    "TrainingImage": "123456789012.dkr.ecr.us-east-1.amazonaws.com/custom-ocr:gpu",
    "TrainingInputMode": "File",
    "MetricDefinitions": [
      {
        "Name": "mAP50",
        "Regex": "all\\s+\\d+\\s+\\d+\\s+[\\d.]+\\s+[\\d.]+\\s+([\\d.]+)"
      }
    ]
  },
  "RoleArn": "arn:aws:iam::123456789012:role/SageMakerExecutionRole",
  "HyperParameters": {
    "weights": "yolov3u.pt",
    "epochs": "100",
    "batch": "8",
    "imgsz": "1024"
  },
  "InputDataConfig": [
    {
      "ChannelName": "training",
      "DataSource": {
        "S3DataSource": {
          "S3DataType": "S3Prefix",
          "S3Uri": "s3://my-custom-ocr-bucket/custom-ocr/datasets/yolo/",
          "S3DataDistributionType": "FullyReplicated"
        }
      }
    }
  ],
  "OutputDataConfig": {
    "S3OutputPath": "s3://my-custom-ocr-bucket/custom-ocr/sagemaker/"
  },
  "Resourc

## Step 5 · Deployment: a serverless endpoint
Serverless inference scales to zero, so an idle endpoint costs **$0**. That suits a tool that reads a few reports at a time. Pass `--instance ml.m5.large` for steady traffic instead.

In [6]:
MODEL_DATA = f"s3://{BUCKET}/custom-ocr/sagemaker/<job-name>/output/model.tar.gz"
run("sm_jobs.py", "--region", REGION, "deploy", "--role", ROLE, "--model-data", MODEL_DATA)
run("sm_jobs.py", "--region", REGION, "invoke", "--image", str(REPO / "samples" / "thyroid_report.jpg"))

[dry-run] sagemaker.create_model(
{
  "ModelName": "custom-ocr-20261025-164534",
  "ExecutionRoleArn": "arn:aws:iam::123456789012:role/SageMakerExecutionRole",
  "PrimaryContainer": {
    "Image": "123456789012.dkr.ecr.us-east-1.amazonaws.com/custom-ocr:cpu",
    "Environment": {
      "OCR_MODE": "tuned",
      "OCR_CONF": "0.25"
    },
    "ModelDataUrl": "s3://my-custom-ocr-bucket/custom-ocr/sagemaker/<job-name>/output/model.tar.gz"
  }
}
)
[dry-run] sagemaker.create_endpoint_config(
{
  "EndpointConfigName": "custom-ocr-cfg-20261025-164534",
  "ProductionVariants": [
    {
      "VariantName": "AllTraffic",
      "ModelName": "custom-ocr-20261025-164534",
      "ServerlessConfig": {
        "MemorySizeInMB": 6144,
        "MaxConcurrency": 2
      }
    }
  ]
}
)
[dry-run] sagemaker.create_endpoint(
{
  "EndpointName": "custom-ocr-endpoint",
  "EndpointConfigName": "custom-ocr-cfg-20261025-164534"
}
)
endpoint ready: custom-ocr-endpoint


[dry-run] sagemaker-runtime.invoke_endpoint(custom-ocr-endpoint, 96119 bytes, image/jpeg)


### The endpoint contract, tested locally
SageMaker starts the container with `serve` and talks to it on port 8080: `GET /ping` must return 200, and `POST /invocations` receives the raw image. The same requests are run here against the app in-process, so the contract is verified without AWS.

In [7]:
from fastapi.testclient import TestClient
from app.server import app
import pandas as pd, io
with TestClient(app) as c:
    print("GET /ping ->", c.get("/ping").status_code)
    r = c.post("/invocations", content=(REPO / "samples" / "liver_report.jpg").read_bytes(),
               headers={"Content-Type": "image/jpeg", "Accept": "text/csv"})
    print("POST /invocations ->", r.status_code)
pd.read_csv(io.StringIO(r.text))[["row", "test_name", "value", "unit", "reference_range", "needs_review"]]

C:\Users\ARKO\Downloads\custom-ocr-lab-reports-main\custom-ocr-lab-reports-main\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


GET /ping -> 200


POST /invocations -> 200


,row,test_name,value,unit,reference_range,needs_review
0,1,ALKALINE PHOSPHATASE,67.00,U/l,45 - 129,False
1,2,BILIRUBIN - TOTAL,0.57,mg/dl,0.3-1.2,True
2,3,BILIRUBIN -DIRECT,0.18,mg/dl,<0.%,True
3,4,BILIRUBIN (INDIRECT),0.39,mg/dl,0-0.9,True
4,5,GAMMA GLUTAMYL TRANSFERASE (GGT),34.30,u/,<55,True
5,6,ASPARTATE AMINOTRANSFERASE (SGOT ),33.10,U/l,<3§,True
6,7,ALANINE TRANSAMINASE (SGPT),38.90,U/l,< 45,True
7,8,PROTEIN - TOTAL,7.14,gm/dl,5.7-8.2,False
8,9,ALBUMIN - SERUM,3.93,gm/dl,3.2-4.8,False
9,10,SERUM ALB/GLOBULIN RATIO,1.22,Ratio,0.9-2,False


## Step 6 · Automation: batch OCR of an S3 folder
Drop photos into `s3://<bucket>/custom-ocr/inbox/`; a SageMaker **Processing job** runs `Custom_OCR.py --image <folder> --excel` inside the same image and writes one CSV + XLSX per photo (and `all_reports.csv`) back to S3. It can be triggered on a schedule (EventBridge) or from an S3 upload event.

In [8]:
run("sm_jobs.py", "--region", REGION, "batch", "--role", ROLE, "--bucket", BUCKET)

[dry-run] sagemaker.create_processing_job(
{
  "ProcessingJobName": "custom-ocr-batch-20261025-164547",
  "RoleArn": "arn:aws:iam::123456789012:role/SageMakerExecutionRole",
  "AppSpecification": {
    "ImageUri": "123456789012.dkr.ecr.us-east-1.amazonaws.com/custom-ocr:cpu",
    "ContainerEntrypoint": [
      "entrypoint.sh",
      "ocr",
      "--image",
      "/opt/ml/processing/input",
      "--out",
      "/opt/ml/processing/output",
      "--excel"
    ]
  },
  "ProcessingInputs": [
    {
      "InputName": "photos",
      "S3Input": {
        "S3Uri": "s3://my-custom-ocr-bucket/custom-ocr/inbox/",
        "LocalPath": "/opt/ml/processing/input",
        "S3DataType": "S3Prefix",
        "S3InputMode": "File"
      }
    }
  ],
  "ProcessingOutputConfig": {
    "Outputs": [
      {
        "OutputName": "csv",
        "S3Output": {
          "S3Uri": "s3://my-custom-ocr-bucket/custom-ocr/results/batch/custom-ocr-batch-20261025-164547/",
          "LocalPath": "/opt/ml/processing/

## Step 7 · Cost control: leave nothing running
Deletes the endpoint, its configs and models, and stops the notebook instance. S3 storage (~0.1 GB) is the only thing left, at under one cent a month.

| Resource | Approx. rate (us-east-1) | This project |
|---|---|---|
| Notebook `ml.t3.medium` | ~$0.05 / h | auto-stops after 60 idle min |
| Training `ml.g4dn.xlarge`, spot | ~$0.22-0.74 / h | ~1 h for YOLOv3 (47 min on a similar laptop GPU) |
| Serverless endpoint | per request + GB-s | $0 when idle |
| Processing `ml.t3.medium` | ~$0.05 / h | ~2 s per photo |
| S3 | ~$0.023 / GB-month | < $0.01 |

**Total for the whole project: well under $1.** Rates are approximate; check the SageMaker pricing page for your region.

In [9]:
run("sm_jobs.py", "--region", REGION, "cleanup")

[dry-run] delete endpoint custom-ocr-endpoint, its endpoint configs and models; stop notebook custom-ocr-notebook
